This script identifies buildings constructed after 2015 by checking whether 2025 building polygons contain the centroids of buildings that already existed in 2015. Buildings without a matching 2015 centroid are classified as new, and residential subtypes are added from the previous classification script.

In [ ]:
import geopandas as gpd
import warnings

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────
PATH_2025 = (
    r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output"
    r"\Analysis_w_LoD2\LoD2\LoD2_2025_classified.gpkg"
)

PATH_2015 = (
    r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Input"
    r"\LoD1_2015\LoD1_2015.gpkg"
)

PATH_RES_TYPES = (
    r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output"
    r"\Analysis_w_LoD2\LoD2\LoD2_2025_residential_types_thr_h11m.gpkg"
)

OUT_NEW = (
    r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output"
    r"\Analysis_w_LoD2\LoD2\LoD2_2025_new_buildings_thr_h11m.gpkg"
)

OUT_EXISTING = (
    r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output"
    r"\Analysis_w_LoD2\LoD2\LoD2_2025_existing_thr_h11m.gpkg"
)

# ─────────────────────────────────────────────
# 1. Load data
# ─────────────────────────────────────────────
print("Loading data...")

gdf_2025 = gpd.read_file(PATH_2025)
gdf_2015 = gpd.read_file(PATH_2015)

print(f"   2025: {len(gdf_2025):,} buildings | CRS: {gdf_2025.crs}")
print(f"   2015: {len(gdf_2015):,} buildings | CRS: {gdf_2015.crs}")

# ─────────────────────────────────────────────
# 2. Harmonize coordinate reference systems
# ─────────────────────────────────────────────
TARGET_CRS = "EPSG:25832"

if gdf_2025.crs != TARGET_CRS:
    print(f"   Reprojecting 2025 data to {TARGET_CRS}")
    gdf_2025 = gdf_2025.to_crs(TARGET_CRS)

if gdf_2015.crs != TARGET_CRS:
    print(f"   Reprojecting 2015 data to {TARGET_CRS}")
    gdf_2015 = gdf_2015.to_crs(TARGET_CRS)

gdf_2025 = gdf_2025.reset_index(drop=True)
gdf_2015 = gdf_2015.reset_index(drop=True)

# ─────────────────────────────────────────────
# 3. Add res_subclass from Script 1
# ─────────────────────────────────────────────
print("\nLoading res_subclass from the residential-types dataset...")

gdf_res_types = gpd.read_file(
    PATH_RES_TYPES,
    columns=["orig_idx", "res_subclass"]
)

res_lookup = gdf_res_types.set_index("orig_idx")["res_subclass"]

gdf_2025["res_subclass"] = None

res_mask = gdf_2025["building_class"] == "residential"

gdf_2025.loc[res_mask, "res_subclass"] = (
    gdf_2025.loc[res_mask].index.map(res_lookup)
)

print(f"   Total residential buildings:       {res_mask.sum():,}")
print(
    f"   Buildings with res_subclass:       "
    f"{gdf_2025['res_subclass'].notna().sum():,}"
)
print(
    f"   Residential subtype not found:     "
    f"{res_mask.sum() - gdf_2025['res_subclass'].notna().sum():,}"
)

# ─────────────────────────────────────────────
# 4. Calculate centroids of the 2015 buildings
# ─────────────────────────────────────────────
print("\nCalculating 2015 building centroids...")

centroids_2015 = gdf_2015[["geometry"]].copy()
centroids_2015["geometry"] = gdf_2015.geometry.centroid
centroids_2015["id_2015"] = centroids_2015.index

print(f"   {len(centroids_2015):,} centroids calculated")

# ─────────────────────────────────────────────
# 5. Spatial join: 2015 centroids to 2025 polygons
# ─────────────────────────────────────────────
print("\nSpatial join: locating 2015 centroids within 2025 building polygons...")

gdf_2025_idx = gdf_2025[["geometry"]].copy()
gdf_2025_idx["id_2025"] = gdf_2025_idx.index

joined = gpd.sjoin(
    centroids_2015,
    gdf_2025_idx,
    how="inner",
    predicate="within"
)

ids_with_2015_centroid = set(joined["id_2025"].unique())

print(
    f"   2025 buildings containing a 2015 centroid "
    f"(existing buildings): {len(ids_with_2015_centroid):,}"
)
print(
    f"   2025 buildings without a 2015 centroid "
    f"(new buildings): {len(gdf_2025) - len(ids_with_2015_centroid):,}"
)

# ─────────────────────────────────────────────
# 6. Filter buildings
# ─────────────────────────────────────────────
is_existing = gdf_2025.index.isin(ids_with_2015_centroid)

gdf_new = gdf_2025[~is_existing].copy()
gdf_existing = gdf_2025[is_existing].copy()

gdf_new["status"] = "new_after_2015"
gdf_existing["status"] = "existing_2015"

# ─────────────────────────────────────────────
# 7. Summary statistics
# ─────────────────────────────────────────────
print("\nResults:")

print(
    f"   Total buildings in 2025: "
    f"{len(gdf_2025):,} buildings (100%)"
)
print(
    f"   Existing buildings in 2015: "
    f"{len(gdf_existing):,} buildings "
    f"({len(gdf_existing) / len(gdf_2025) * 100:.1f}%)"
)
print(
    f"   New buildings: "
    f"{len(gdf_new):,} buildings "
    f"({len(gdf_new) / len(gdf_2025) * 100:.1f}%)"
)

if "building_class" in gdf_new.columns:
    print("\n   New buildings by building_class:")

    for cls, count in gdf_new["building_class"].value_counts().items():
        percentage = count / len(gdf_new) * 100
        print(f"     {cls:<20} {count:>6} ({percentage:.1f}%)")

if "res_subclass" in gdf_new.columns:
    print("\n   New buildings by res_subclass (residential buildings only):")

    residential_new_count = gdf_new["res_subclass"].notna().sum()

    for cls, count in gdf_new["res_subclass"].dropna().value_counts().items():
        percentage = count / residential_new_count * 100
        print(f"     {cls:<20} {count:>6} ({percentage:.1f}%)")

# ─────────────────────────────────────────────
# 8. Save results
# ─────────────────────────────────────────────
print("\nSaving output files...")

gdf_new.to_file(OUT_NEW, driver="GPKG")
print(f"   New buildings:      {OUT_NEW}")

gdf_existing.to_file(OUT_EXISTING, driver="GPKG")
print(f"   Existing buildings: {OUT_EXISTING}")

print("\nDone!")
```
